# Precariedade habitacional nos setores censitários da Região Metropolitana de São Paulo

**Projeto Final – Introdução à Ciência de Dados (2026)**

**Grupo:** [nome completo dos integrantes]

---

## Introdução

### Conjunto de dados
- **Fonte:** IBGE, Censo Demográfico 2022, Agregados por Setores Censitários (tabelas Básico e Características dos Domicílios 1, 2 e 3). Link: [inserir link da página do IBGE]
- **Recorte:** setores censitários dos 39 municípios da Região Metropolitana de São Paulo (RMSP).
- **Unidade de análise:** setor censitário.
- A base foi consolidada no notebook `BD_setores_RMSP.ipynb`.

### Objetivo
Explorar as condições de moradia dos domicílios nos setores censitários da RMSP (acesso à água, esgotamento sanitário, destino do lixo e tipo de moradia) e identificar grupos de setores com perfis semelhantes de precariedade.

### Perguntas de pesquisa
1. Como as carências de água, esgoto, lixo e moradia se distribuem entre os setores da RMSP?
2. Essas carências ocorrem juntas nos mesmos setores?
3. Quais tipos de precariedade habitacional podem ser identificados entre os setores?
4. Setores de um mesmo município pertencem a tipos diferentes de precariedade?

## Metodologia

### 1. Carregamento e descrição da base

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler

ARQ_BASE = "setores_rmsp.csv"   # gerado pelo notebook BD_setores_RMSP
base = pd.read_csv(ARQ_BASE, sep=";", encoding="utf-8", dtype={"CD_SETOR": str}, low_memory=False)
base.columns = [c if not c.lower().startswith("v0") or len(c) > 5 else c.lower() for c in base.columns]

print(f"Setores (tuplas): {base.shape[0]:,}")
print(f"Colunas (atributos): {base.shape[1]}")
print(f"Municípios: {base['CD_MUN'].nunique()}")

### 2. Seleção das variáveis

As variáveis foram selecionadas pelos mesmos critérios do estudo na escala municipal: pertencer às dimensões da moradia adequada (água, esgoto e banheiro, durabilidade da moradia e resíduos sólidos), indicar privação (e não a presença do serviço) e referir-se ao total de domicílios.

In [ ]:
VARIAVEIS = {
    # Durabilidade da moradia (domicílios permanentes)
    "V00050": "Casa de cômodos ou cortiço",
    "V00051": "Habitação indígena sem paredes ou maloca",
    "V00052": "Estrutura residencial degradada ou inacabada",
    # Domicílios improvisados
    "V00053": "Improvisado: tenda ou barraca",
    "V00054": "Improvisado: dentro de estabelecimento",
    "V00055": "Improvisado: abrigo natural",
    "V00056": "Improvisado: estrutura em logradouro público",
    "V00057": "Improvisado: estrutura não residencial degradada",
    "V00058": "Improvisado: veículo",
    # Abastecimento de água
    "V00113": "Água: poço raso, freático ou cacimba",
    "V00114": "Água: fonte, nascente ou mina",
    "V00115": "Água: carro-pipa",
    "V00116": "Água: chuva armazenada",
    "V00117": "Água: rios, açudes, córregos, lagos",
    "V00118": "Água: outra forma",
    "V00200": "Água encanada só até o terreno",
    "V00201": "Água não chega encanada",
    "V00464": "Sem ligação à rede geral de água",
    # Esgoto e banheiro
    "V00236": "Banheiro de uso comum",
    "V00237": "Apenas sanitário ou buraco",
    "V00312": "Esgoto: fossa rudimentar ou buraco",
    "V00313": "Esgoto: vala",
    "V00314": "Esgoto: rio, lago, córrego ou mar",
    "V00315": "Esgoto: outra forma",
    "V00316": "Sem banheiro nem sanitário",
    # Resíduos sólidos
    "V00399": "Lixo queimado",
    "V00400": "Lixo enterrado",
    "V00401": "Lixo em terreno baldio ou área pública",
    "V00402": "Lixo: outro destino",
}
IMPROVISADOS = ["V00053", "V00054", "V00055", "V00056", "V00057", "V00058"]
DENOMINADORES = ["V00001", "v0007"]   # domicílios permanentes ocupados | total de domicílios ocupados
selecionadas = list(VARIAVEIS)
faltando = [v for v in selecionadas + DENOMINADORES if v not in base.columns]
print("Variáveis não encontradas na base:", faltando if faltando else "nenhuma")

### 3. Preparação dos dados

#### 3.1 Setores sem dados de domicílios
Setores sem nenhuma informação nas tabelas de domicílios (sem domicílios ocupados ou com todos os dados omitidos por sigilo) são excluídos.

In [ ]:
dom_cols = [c for c in base.columns if c.startswith("V")]
sem_dados = base[dom_cols].isna().all(axis=1)
print(f"Setores sem dados de domicílios (excluídos): {sem_dados.sum():,}")
df = base[~sem_dados].copy()
print(f"Setores na análise: {len(df):,}")
print(f"Domicílios por setor (V00001): mínimo {df['V00001'].min():.0f} | mediana {df['V00001'].median():.0f} | máximo {df['V00001'].max():.0f}")

#### 3.2 Valores ausentes
O IBGE omite, por sigilo, as contagens de 1 e 2 domicílios. Nos dados, o menor valor diferente de zero é sempre 3, o que confirma a regra. Os valores ausentes são, portanto, substituídos por 1,5, ponto médio entre as duas possibilidades.

In [ ]:
ausentes = pd.DataFrame({
    "% de setores com ausente": (df[selecionadas].isna().mean() * 100).round(1),
    "Menor valor > 0": df[selecionadas].where(df[selecionadas] > 0).min(),
})
ausentes.index = ausentes.index.map(VARIAVEIS)
display(ausentes)
print(f"Setores com ausente em pelo menos uma variável: {df[selecionadas].isna().any(axis=1).mean()*100:.1f}%")

df[selecionadas] = df[selecionadas].fillna(1.5)

#### 3.3 Conversão em taxas
As contagens são convertidas em porcentagem, para comparar setores de tamanhos diferentes. As variáveis dos domicílios permanentes são divididas pelo total de domicílios permanentes ocupados (V00001), e os tipos de domicílio improvisado, pelo total de domicílios ocupados (v0007).

Também se verificam inconsistências: taxas acima de 100% ou denominadores iguais a zero.

In [ ]:
taxas = pd.DataFrame({"CD_SETOR": df["CD_SETOR"], "CD_MUN": df["CD_MUN"], "NM_MUN": df["NM_MUN"]})
for v in selecionadas:
    den = df["v0007"] if v in IMPROVISADOS else df["V00001"]
    taxas[v] = df[v] / den.replace(0, np.nan) * 100

print(f"Taxas acima de 100% (inconsistências): {(taxas[selecionadas] > 100).sum().sum()}")
print(f"Taxas sem denominador válido: {taxas[selecionadas].isna().sum().sum()}")
taxas[selecionadas] = taxas[selecionadas].clip(upper=100).fillna(0)

#### 3.4 Variáveis de baixa ocorrência
Condições presentes em muito poucos setores não formam padrões e, após a normalização, ganhariam peso desproporcional. Variáveis com ocorrência em menos de 1% dos setores são analisadas à parte (variáveis suplementares).

In [ ]:
PCT_MIN = 1.0   # % mínima de setores com ocorrência
ocorrencia = pd.DataFrame({
    "Descrição": [VARIAVEIS[v] for v in selecionadas],
    "% de setores com ocorrência": [(taxas[v] > 0).mean() * 100 for v in selecionadas],
    "Máximo (%)": [taxas[v].max() for v in selecionadas],
}, index=selecionadas).round(2).sort_values("% de setores com ocorrência")
display(ocorrencia)

suplementares = ocorrencia[ocorrencia["% de setores com ocorrência"] < PCT_MIN].index.tolist()
print("Variáveis suplementares:", [VARIAVEIS[v] for v in suplementares])

#### 3.5 Redundância entre as variáveis
Pares de taxas com correlação |r| ≥ 0,85. Retira-se a variável quando uma contém a outra (por exemplo, a V00464, que é o total das formas de abastecimento fora da rede).

In [ ]:
candidatas = [v for v in selecionadas if v not in suplementares]
corr = taxas[candidatas].corr()
pares = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().reset_index()
pares.columns = ["A", "B", "r"]
display(pares[pares["r"].abs() >= 0.85].assign(A=lambda d: d["A"].map(VARIAVEIS), B=lambda d: d["B"].map(VARIAVEIS)).round(2))

REDUNDANTES = ["V00464"]   # ajuste após ver os pares acima
variaveis_modelo = [v for v in candidatas if v not in REDUNDANTES]
print(f"Variáveis no agrupamento: {len(variaveis_modelo)}")

#### 3.6 Outliers
Os valores atípicos são identificados nos boxplots (Visualização 1). Eles correspondem a situações reais de precariedade e, por isso, são mantidos. Porém, como a normalização Min-Max usa o valor máximo de cada variável, os valores acima do percentil 99 são limitados a esse percentil, para que poucos setores não definam a escala da variável inteira.

In [ ]:
X = taxas[variaveis_modelo].copy()
p99 = X.quantile(0.99)
print(f"Valores limitados ao percentil 99: {(X > p99).sum().sum():,}")
X = X.clip(upper=p99, axis=1)

#### 3.7 Normalização Min-Max
Transforma cada variável para a escala de 0 a 1, para que todas tenham o mesmo peso no agrupamento.

In [ ]:
X_norm = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=variaveis_modelo, index=X.index)
X_norm.describe().T[["min", "max", "mean"]].round(3)

### 4. Análise descritiva

#### 4.1 Análise de uma variável
Frequência, valores mínimo e máximo, medidas de centralidade e de dispersão e assimetria das taxas (% de domicílios).

In [ ]:
desc = pd.DataFrame({
    "% setores com ocorrência": (taxas[variaveis_modelo] > 0).mean() * 100,
    "Mínimo": taxas[variaveis_modelo].min(),
    "Máximo": taxas[variaveis_modelo].max(),
    "Média": taxas[variaveis_modelo].mean(),
    "Mediana": taxas[variaveis_modelo].median(),
    "Desvio-padrão": taxas[variaveis_modelo].std(),
    "Assimetria": taxas[variaveis_modelo].skew(),
}).round(2)
desc.index = desc.index.map(VARIAVEIS)
display(desc.sort_values("Média", ascending=False))

#### Visualização 1 — Como se distribuem as principais carências entre os setores? (pergunta 1)
Boxplots das variáveis com maior média, mostrando assimetria e valores atípicos.

In [ ]:
principais = desc.sort_values("Média", ascending=False).index[:8]
cod = {d: v for v, d in VARIAVEIS.items()}
dados_box = taxas[[cod[d] for d in principais]].rename(columns=VARIAVEIS)
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=dados_box, orient="h", ax=ax, fliersize=1.5, color="#9ECAE1")
ax.set_xlabel("% dos domicílios do setor")
ax.set_title("Distribuição das principais carências nos setores da RMSP")
plt.tight_layout(); plt.show()

#### 4.2 Análise de duas variáveis

#### Visualização 2 — As carências ocorrem juntas nos mesmos setores? (pergunta 2)
Correlação de Spearman entre as taxas (menos sensível a valores extremos e a distribuições assimétricas).

In [ ]:
corr_s = taxas[variaveis_modelo].corr(method="spearman").rename(index=VARIAVEIS, columns=VARIAVEIS)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_s, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, ax=ax,
            cbar_kws={"label": "Correlação de Spearman"})
ax.set_title("Correlação entre as carências nos setores da RMSP")
plt.tight_layout(); plt.show()

### 5. Agrupamento (K-Means)

#### 5.1 Escolha do número de grupos
Método do cotovelo (inércia) e coeficiente de silhueta. Com mais de 45 mil setores, a silhueta é calculada numa amostra aleatória de 5 mil setores.

In [ ]:
amostra = X_norm.sample(5000, random_state=42).index
resultados = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_norm)
    sil = silhouette_score(X_norm.loc[amostra], km.labels_[X_norm.index.get_indexer(amostra)])
    resultados.append({"k": k, "Inércia": km.inertia_, "Silhueta (amostra)": sil})
res_k = pd.DataFrame(resultados).round(3)
display(res_k)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(res_k["k"], res_k["Inércia"], "o-", color="#08519C"); ax1.set_xlabel("Número de grupos (k)"); ax1.set_ylabel("Inércia", color="#08519C")
ax2 = ax1.twinx(); ax2.plot(res_k["k"], res_k["Silhueta (amostra)"], "s--", color="#CB181D"); ax2.set_ylabel("Silhueta", color="#CB181D")
ax1.set_title("Escolha do número de grupos"); plt.tight_layout(); plt.show()

#### 5.2 Modelo final
Defina o `K_FINAL` depois de observar o cotovelo e a silhueta.

In [ ]:
K_FINAL = 3   # ajuste após observar a tabela e o gráfico acima
km = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10).fit(X_norm)
taxas["grupo"] = km.labels_ + 1
print(taxas["grupo"].value_counts().sort_index().rename("setores por grupo").to_string())

## Resultados

### Perfil dos grupos (pergunta 3)
Média de cada variável (% dos domicílios) em cada grupo, ordenada pela diferença entre os grupos.

In [ ]:
perfil = taxas.groupby("grupo")[variaveis_modelo].mean().T
perfil["Diferença entre grupos"] = perfil.std(axis=1)
perfil = perfil.sort_values("Diferença entre grupos", ascending=False).round(2)
perfil.index = perfil.index.map(VARIAVEIS)
display(perfil)

#### Visualização 3 — Como os grupos se diferenciam nas principais carências? (pergunta 3)

In [ ]:
top = [cod[d] for d in perfil.index[:2]]
amostra_plot = taxas.sample(min(8000, len(taxas)), random_state=1)
fig, ax = plt.subplots(figsize=(9, 7))
sns.scatterplot(data=amostra_plot, x=top[0], y=top[1], hue="grupo", palette="Set2", s=12, alpha=0.6, ax=ax)
ax.set_xlabel(f"{VARIAVEIS[top[0]]} (%)"); ax.set_ylabel(f"{VARIAVEIS[top[1]]} (%)")
ax.set_title("Setores segundo as duas carências que mais diferenciam os grupos (amostra de 8 mil setores)")
plt.tight_layout(); plt.show()

#### Visualização 4 — Onde estão os setores de cada grupo? (pergunta 3)
Requer a malha de setores censitários de 2022 do IBGE (arquivo `.zip` ou `.shp`). Ajuste o caminho abaixo.

In [ ]:
import os
import geopandas as gpd

ARQ_MALHA = "data/SP_setores_CD2022.zip"   # malha de setores censitários 2022 (IBGE); ajuste o nome
if not os.path.exists(ARQ_MALHA):
    print("Malha de setores não encontrada. Ajuste ARQ_MALHA e rode esta célula de novo.")
else:
    malha = gpd.read_file(ARQ_MALHA)
    col = next(c for c in malha.columns if "setor" in c.lower())
    malha = malha.rename(columns={col: "CD_SETOR"})
    malha["CD_SETOR"] = malha["CD_SETOR"].astype(str).str.strip()
    mapa = malha.merge(taxas[["CD_SETOR", "grupo"]], on="CD_SETOR", how="inner")
    fig, ax = plt.subplots(figsize=(12, 9))
    mapa.plot(column="grupo", categorical=True, cmap="Set2", legend=True, linewidth=0, ax=ax,
              legend_kwds={"title": "Grupo", "loc": "lower right"})
    ax.set_axis_off(); ax.set_title("Grupos de precariedade habitacional por setor censitário – RMSP")
    plt.tight_layout(); plt.show()

#### Visualização 5 — Setores de um mesmo município pertencem a tipos diferentes de precariedade? (pergunta 4)

In [ ]:
comp = pd.crosstab(taxas["NM_MUN"], taxas["grupo"], normalize="index") * 100
comp = comp.sort_values(comp.columns[0])
fig, ax = plt.subplots(figsize=(11, 12))
comp.plot(kind="barh", stacked=True, colormap="Set2", ax=ax, width=0.85)
ax.set_xlabel("% dos setores do município"); ax.set_ylabel("")
ax.set_title("Composição dos municípios da RMSP segundo os grupos de setores")
ax.legend(title="Grupo", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

### Variáveis suplementares
Ocorrência das condições de baixa ocorrência em cada grupo.

In [ ]:
if suplementares:
    sup = pd.DataFrame({VARIAVEIS[v]: taxas.groupby("grupo")[v].apply(lambda s: (s > 0).sum()) for v in suplementares}).T
    sup.columns = [f"Grupo {g} (setores com ocorrência)" for g in sup.columns]
    display(sup)
else:
    print("Nenhuma variável suplementar com o critério atual.")

## Discussão dos resultados

*[Escrever aqui as respostas às quatro perguntas de pesquisa, com base nas tabelas e visualizações acima.]*

1. **Distribuição das carências:** ...
2. **Carências que ocorrem juntas:** ...
3. **Tipos de precariedade:** ...
4. **Diferenças dentro dos municípios:** ...